# Module 3 · Explore · Practitioner Notebook
**Habeeba Siddiqui · outpatient_visits.csv**

This is the notebook Habeeba walks through in the Explore video. Each prompt she gave her AI tool is in a text cell, followed by the code it returned and the output from running it. Compare it with your own exploration after you have finished yours.

The ground rules
Paste this at the start of any AI session for this module. It covers how the AI should work with you, whatever the task is, and it is a decent example of a careful prompt.
 *I'm a student learning data analysis. I'll be asking you to help me write Python and pandas code for a dataset I'm working with. Here is how I want you to work with me. - If what I ask for is ambiguous, ask me a question instead of guessing. - Write the code I ask for. Don't decide what the analysis should be. - Keep the code simple and standard. Use the plain, common way to do something rather than a clever one. - Comment every line with what it does, so I can follow the code without already knowing pandas.*

 AI promt: I have a CSV file called outpatient_visits.csv. Load it and tell me: 1. How many rows and columns it has 2. Every column name, its data type, and how many non-null values it has 3.Print some sample rows, so I can see what the data actually looks like.
 Do not do any analysis yet. Just load and describe.

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("https://raw.githubusercontent.com/EDC-AICC/ai-ds-development/main/src/data/outpatient_visits.csv")

# 1. Show the number of rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# 2. Show each column name, data type, and non-null count
print("\nDataset information:")
df.info()

# 3. Show five sample rows
print("\nSample rows:")
print(df.head())

Number of rows: 812
Number of columns: 13

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 812 entries, 0 to 811
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   visit_id            812 non-null    object 
 1   patient_id          812 non-null    object 
 2   visit_date          812 non-null    object 
 3   icd_code            783 non-null    object 
 4   icd_description     783 non-null    object 
 5   visit_type          812 non-null    object 
 6   gender              812 non-null    object 
 7   age                 779 non-null    float64
 8   insurance_type      812 non-null    object 
 9   provider_id         812 non-null    object 
 10  county              564 non-null    object 
 11  copay_amount        768 non-null    float64
 12  follow_up_required  812 non-null    object 
dtypes: float64(2), object(11)
memory usage: 82.6+ KB

Sample rows:
  visit_id patient_id  visit

Dataset: outpatient_visits.csv — 812 rows, 13 columns.
Brief: who uses the clinic and how often, repeat visitors
and uninsured patients, for the board and county funder.


Already know: visit_date is object (not parsed as date),
county has 248 nulls (30%), age 33 nulls,
copay_amount 44 nulls.


Write readable step-by-step code, not chained one-liners.
Do not modify the data.

Run these checks:


1. patient_id — unique count, length range,
   any whitespace or case inconsistencies


2. visit_date — parse with to_datetime(errors='coerce'),
   count failures, show date range, flag records
   outside last 2 years


3. insurance_type — all distinct values and counts,
   flag any that differ only by case or whitespace


4. Duplicates — visit_id appearing more than once;
   patient_id + visit_date appearing more than once


5. county nulls — break down the 248 missing values
   by insurance_type and by visit_type


6. age and copay_amount — min, max, mean; flag
   values below 0 or above 110 for age, negatives
   for copay; check if missing age concentrates
   in any insurance group



In [2]:
import pandas as pd

# --------------------------------------------------
# DATA QUALITY CHECKS
# Dataset: outpatient_visits.csv
# --------------------------------------------------

print("DATA QUALITY CHECKS")
print("=" * 50)


# --------------------------------------------------
# 1. PATIENT_ID
# --------------------------------------------------

print("\n1. PATIENT_ID CHECKS")

patient_ids = df["patient_id"].astype("string")

print("Unique patients:", df["patient_id"].nunique())

id_lengths = patient_ids.dropna().str.len()

print("Shortest patient_id length:", id_lengths.min())
print("Longest patient_id length:", id_lengths.max())

# Check for extra spaces
patient_ids_no_spaces = patient_ids.str.strip()
whitespace_issues = patient_ids != patient_ids_no_spaces

print("Patient IDs with extra whitespace:", whitespace_issues.sum())

# Check for IDs that differ only by case or whitespace
normalized_patient_ids = patient_ids.str.strip().str.lower()

patient_id_check = pd.DataFrame({
    "original": patient_ids,
    "normalized": normalized_patient_ids
})

patient_variants = (
    patient_id_check
    .groupby("normalized")["original"]
    .nunique()
)

patient_id_issues = patient_variants[patient_variants > 1]

print("Patient IDs with case or whitespace inconsistencies:")
print(patient_id_issues)


# --------------------------------------------------
# 2. VISIT_DATE
# --------------------------------------------------

print("\n2. VISIT_DATE CHECKS")

# Parse dates in a new variable so df is not changed
parsed_dates = pd.to_datetime(
    df["visit_date"],
    errors="coerce"
)

# Count dates that failed to convert
failed_dates = (
    df["visit_date"].notna()
    & parsed_dates.isna()
)

print("Dates that could not be parsed:", failed_dates.sum())

print("Earliest visit date:", parsed_dates.min())
print("Latest visit date:", parsed_dates.max())

# Use latest date in dataset as the reference point
latest_date = parsed_dates.max()
two_year_cutoff = latest_date - pd.DateOffset(years=2)

outside_two_years = parsed_dates < two_year_cutoff

print("Two-year cutoff:", two_year_cutoff)
print("Visits outside the last two years:", outside_two_years.sum())

if outside_two_years.any():
    print("\nRecords outside the last two years:")
    print(
        df.loc[
            outside_two_years,
            ["patient_id", "visit_date"]
        ]
    )


# --------------------------------------------------
# 3. INSURANCE_TYPE
# --------------------------------------------------

print("\n3. INSURANCE_TYPE CHECKS")

insurance_counts = df["insurance_type"].value_counts(
    dropna=False
)

print("Insurance type counts:")
print(insurance_counts)

insurance_original = df["insurance_type"].astype("string")

insurance_normalized = (
    insurance_original
    .str.strip()
    .str.lower()
)

insurance_check = pd.DataFrame({
    "original": insurance_original,
    "normalized": insurance_normalized
})

insurance_variants = (
    insurance_check
    .groupby("normalized")["original"]
    .nunique()
)

insurance_issues = insurance_variants[
    insurance_variants > 1
]

print("\nInsurance categories with case or whitespace inconsistencies:")
print(insurance_issues)


# --------------------------------------------------
# 4. DUPLICATES
# --------------------------------------------------

print("\n4. DUPLICATE CHECKS")

# Duplicate visit_id
duplicate_visit_id = df["visit_id"].duplicated(
    keep=False
)

print(
    "Rows with duplicated visit_id:",
    duplicate_visit_id.sum()
)

if duplicate_visit_id.any():
    print("\nDuplicate visit_id records:")
    print(
        df.loc[
            duplicate_visit_id,
            ["visit_id", "patient_id", "visit_date"]
        ].sort_values("visit_id")
    )


# Duplicate patient_id + visit_date
patient_date_check = pd.DataFrame({
    "patient_id": df["patient_id"],
    "visit_date": parsed_dates
})

duplicate_patient_date = patient_date_check.duplicated(
    subset=["patient_id", "visit_date"],
    keep=False
)

print(
    "Rows with duplicate patient_id + visit_date:",
    duplicate_patient_date.sum()
)

if duplicate_patient_date.any():
    print("\nDuplicate patient/date records:")
    print(
        df.loc[
            duplicate_patient_date,
            ["visit_id", "patient_id", "visit_date"]
        ]
    )


# --------------------------------------------------
# 5. COUNTY MISSING VALUES
# --------------------------------------------------

print("\n5. COUNTY MISSING VALUE CHECKS")

county_missing = df["county"].isna()

print("Missing county values:", county_missing.sum())

county_missing_percent = county_missing.mean() * 100

print(
    "Percent missing county:",
    round(county_missing_percent, 1),
    "%"
)

missing_county_data = df.loc[county_missing]

print("\nMissing county values by insurance type:")

missing_by_insurance = (
    missing_county_data["insurance_type"]
    .value_counts(dropna=False)
)

print(missing_by_insurance)


print("\nMissing county values by visit type:")

missing_by_visit_type = (
    missing_county_data["visit_type"]
    .value_counts(dropna=False)
)

print(missing_by_visit_type)


# --------------------------------------------------
# 6. AGE
# --------------------------------------------------

print("\n6. AGE CHECKS")

print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())
print("Mean age:", round(df["age"].mean(), 1))
print("Missing age:", df["age"].isna().sum())

invalid_age = (
    (df["age"] < 0)
    | (df["age"] > 110)
)

print("Invalid age values:", invalid_age.sum())

if invalid_age.any():
    print("\nRecords with invalid ages:")
    print(
        df.loc[
            invalid_age,
            ["patient_id", "age"]
        ]
    )


# Check whether missing age is concentrated
# in particular insurance groups

insurance_total = (
    df.groupby(
        "insurance_type",
        dropna=False
    )
    .size()
)

missing_age_by_insurance = (
    df.loc[df["age"].isna()]
    .groupby(
        "insurance_type",
        dropna=False
    )
    .size()
)

age_missing_check = pd.DataFrame({
    "total_records": insurance_total,
    "missing_age": missing_age_by_insurance
})

age_missing_check["missing_age"] = (
    age_missing_check["missing_age"]
    .fillna(0)
)

age_missing_check["percent_missing_age"] = (
    age_missing_check["missing_age"]
    / age_missing_check["total_records"]
    * 100
)

print("\nMissing age by insurance group:")
print(age_missing_check.round(1))


# --------------------------------------------------
# 7. COPAY_AMOUNT
# --------------------------------------------------

print("\n7. COPAY_AMOUNT CHECKS")

print(
    "Minimum copay:",
    df["copay_amount"].min()
)

print(
    "Maximum copay:",
    df["copay_amount"].max()
)

print(
    "Mean copay:",
    round(df["copay_amount"].mean(), 2)
)

print(
    "Missing copay:",
    df["copay_amount"].isna().sum()
)

negative_copay = df["copay_amount"] < 0

print(
    "Negative copay values:",
    negative_copay.sum()
)

if negative_copay.any():
    print("\nRecords with negative copay amounts:")
    print(
        df.loc[
            negative_copay,
            ["patient_id", "copay_amount"]
        ]
    )


print("\n" + "=" * 50)
print("DATA QUALITY CHECKS COMPLETE")

DATA QUALITY CHECKS

1. PATIENT_ID CHECKS
Unique patients: 200
Shortest patient_id length: 4
Longest patient_id length: 4
Patient IDs with extra whitespace: 0
Patient IDs with case or whitespace inconsistencies:
Series([], Name: original, dtype: int64)

2. VISIT_DATE CHECKS
Dates that could not be parsed: 545
Earliest visit date: 2022-01-02 00:00:00
Latest visit date: 2023-12-31 00:00:00
Two-year cutoff: 2021-12-31 00:00:00
Visits outside the last two years: 0

3. INSURANCE_TYPE CHECKS
Insurance type counts:
insurance_type
mcd           88
medicaid      82
MEDICAID      80
Medicaid      56
MEDICARE      55
Private       55
Commercial    51
Medicare      51
PRIVATE       46
private       45
self-pay      38
UNINSURED     38
MCR           36
medicare      33
uninsured     18
commercial    15
Self-Pay      15
Uninsured     10
Name: count, dtype: int64

Insurance categories with case or whitespace inconsistencies:
normalized
commercial    2
medicaid      3
medicare      3
private       3
s

Here are the findings from my data quality checks
on outpatient_visits.csv:

[paste PROBLEMS FOUND output here]

My brief is: who uses the clinic and how often,
centred on repeat visitors and uninsured patients,
for the board and county funder.

For each problem listed, explain in one sentence
why it matters for this specific brief — what
breaks or becomes unreliable if it is not fixed
before the analysis runs.

If a problem does not affect any metric in the
brief, say so explicitly.

The AI findings on the output
| Column / Area               | Problem                                                                                                                               | Why it matters for this brief                                                                                                                                                                  |
| --------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `visit_date`                | 545 dates were reported as failing to parse.                                                                                          | Repeat-visitor analysis depends on correctly ordering visits over time, so unparsed dates could make visit frequency and return patterns unreliable.                                           |
| `insurance_type`            | The same insurance categories appear with different capitalization, such as `Medicaid`, `MEDICAID`, and `medicaid`.                   | These variations could split one insurance group into several categories and produce an incorrect count or percentage of uninsured and insured patients.                                       |
| `insurance_type`            | Additional labels such as `mcd`, `MCR`, and `self-pay` may represent existing insurance categories but need their meanings confirmed. | The definition of uninsured must be consistent before calculating the uninsured share for the board or county funder.                                                                          |
| `visit_id`                  | 30 rows have duplicated `visit_id` values.                                                                                            | If these represent duplicated visits, they could inflate the total number of visits and make some patients appear to use the clinic more often than they actually did.                         |
| `patient_id + visit_date`   | 520 rows share the same patient ID and visit date with another record.                                                                | These records need to be checked before treating them as duplicates because multiple legitimate visits could occur on the same day; removing them automatically could undercount clinic use.   |
| `county`                    | 248 records, or 30.5% of the dataset, are missing county information.                                                                 | A large amount of missing county data could make geographic results incomplete or biased, which is especially important for the county funder.                                                 |
| `county` + `insurance_type` | Missing county values occur across insurance groups, but the insurance labels are currently inconsistent.                             | Until insurance categories are standardized, it is difficult to tell whether county information is disproportionately missing for uninsured patients.                                          |
| `age`                       | 10 records contain impossible ages, including `-5` and `999`.                                                                         | These values could distort any description of who uses the clinic if age is included in the board's patient profile, but they do not directly affect repeat-visitor or uninsured calculations. |
| `age`                       | 33 age values are missing.                                                                                                            | Missing age could affect demographic descriptions of clinic users, but it does not directly affect the main repeat-visitor or uninsured measures.                                              |
| `copay_amount`              | 14 records have negative copay amounts.                                                                                               | Copay is not part of the current brief, so this issue does not directly affect the main analysis unless copay is later used as a measure.                                                      |
